# 16. Statistical analysis — per-scene metrics, bootstrap CIs, Wilcoxon tests

Recomputes per-scene Dice for the four base models and the weighted ensemble
on the test split, writes `results/per_scene_dice.csv`, and reproduces the
95% bootstrap confidence intervals (10,000 resamples of the test scenes) and
the paired Wilcoxon signed-rank tests reported in Table D.

Run this after notebooks 01-13 have exported their test-set binary predictions.


## 1. Configuration

In [ ]:
import os, re, glob, json
import numpy as np
import pandas as pd
import cv2
from scipy.stats import wilcoxon

# Binary test predictions exported by the individual model notebooks and by
# the ensemble notebook (13). Keys are the column names in the output CSV.
PRED_DIRS = {
    "UNet-ResNet50":  "test_masks/unet_with_resnet50",
    "DeepLabv3+-ResNet50": "test_masks/dlv3+_with_resnet50",
    "YOLO":           "test_masks/yolo",
    "Mask2Former":    "test_masks/mask2former",
    "Ensemble":       "ensemble_test_probability/weighted_by_dice",
}

GT_DIR = "split_data/test/Masks"
RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)

N_BOOTSTRAP = 10000
RNG_SEED = 42

# Map a scene file name to its evaluation region by substring. Adjust the
# keywords to match your file-naming convention if needed.
REGION_KEYWORDS = {
    "Sulak & Terek": ("sulak", "terek"),
    "Mzymta":        ("mzymta",),
    "Vistula & Kaliningrad Bay": ("vistula", "kaliningrad", "pregolya", "baltiysk"),
    "Ural":          ("ural",),
    "Kura":          ("kura",),
}

def region_of(name):
    low = name.lower()
    for region, keys in REGION_KEYWORDS.items():
        if any(k in low for k in keys):
            return region
    return "Unassigned"


## 2. Helpers

In [ ]:
def load_binary(path, ref_shape=None):
    """Load a 0/255 mask as a 0/1 uint8 array, optionally resized (nearest)."""
    m = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if m is None:
        return None
    if ref_shape is not None and m.shape != ref_shape:
        m = cv2.resize(m, (ref_shape[1], ref_shape[0]),
                       interpolation=cv2.INTER_NEAREST)
    return (m > 127).astype(np.uint8)

def dice(pred, gt):
    inter = int(np.sum(pred * gt))
    denom = int(np.sum(pred) + np.sum(gt))
    return 1.0 if denom == 0 else 2.0 * inter / denom

def match_gt(pred_name, gt_names):
    """Match a prediction file to its ground-truth mask by name, then by date."""
    base = pred_name.replace("_pred.png", ".png")
    if base in gt_names:
        return base
    if pred_name in gt_names:
        return pred_name
    m = re.search(r"(\d{4}-\d{2}-\d{2})", pred_name)
    if m:
        for g in gt_names:
            if m.group(1) in g:
                return g
    return None


## 3. Per-scene Dice table

In [ ]:
gt_names = sorted(os.listdir(GT_DIR))
gt_set = set(gt_names)

# Reference scene list = ground-truth masks of the test split.
records = {}
for g in gt_names:
    records[g] = {"scene": g, "region": region_of(g)}

for model, pdir in PRED_DIRS.items():
    if not os.path.isdir(pdir):
        print(f"[warn] missing prediction dir for {model}: {pdir}")
        continue
    for pf in sorted(os.listdir(pdir)):
        if not pf.lower().endswith((".png", ".jpg", ".jpeg")):
            continue
        gname = match_gt(pf, gt_set)
        if gname is None:
            continue
        gt = load_binary(os.path.join(GT_DIR, gname))
        pred = load_binary(os.path.join(pdir, pf), ref_shape=gt.shape)
        if pred is None:
            continue
        records[gname][model] = dice(pred, gt)

df = pd.DataFrame(list(records.values()))
model_cols = [m for m in PRED_DIRS if m in df.columns]
df = df.dropna(subset=model_cols)  # keep scenes scored by every model
csv_path = os.path.join(RESULTS_DIR, "per_scene_dice.csv")
df.to_csv(csv_path, index=False)
print(f"{len(df)} test scenes scored by all {len(model_cols)} models -> {csv_path}")
df.head()


## 4. Per-region and overall Dice (reproduces the body of Table D)

In [ ]:
region_order = ["Sulak & Terek", "Mzymta", "Vistula & Kaliningrad Bay",
                "Ural", "Kura"]
summary = []
for region in region_order:
    sub = df[df["region"] == region]
    if len(sub) == 0:
        continue
    row = {"Region": f"{region} (n={len(sub)})"}
    row.update({m: round(sub[m].mean(), 3) for m in model_cols})
    summary.append(row)
overall = {"Region": f"Overall (n={len(df)})"}
overall.update({m: round(df[m].mean(), 3) for m in model_cols})
summary.append(overall)
pd.DataFrame(summary)


## 5. Bootstrap 95% CIs and paired comparisons vs the ensemble

In [ ]:
rng = np.random.default_rng(RNG_SEED)
scores = {m: df[m].to_numpy() for m in model_cols}
n = len(df)
idx = np.arange(n)

def boot_ci(values, stat=np.mean):
    samples = np.empty(N_BOOTSTRAP)
    for b in range(N_BOOTSTRAP):
        take = rng.integers(0, n, n)
        samples[b] = stat(values[take])
    return np.percentile(samples, [2.5, 97.5])

print("Overall Dice with 95% bootstrap CI:")
for m in model_cols:
    lo, hi = boot_ci(scores[m])
    print(f"  {m:24s} {scores[m].mean():.3f}  [{lo:.3f}, {hi:.3f}]")

print("\nEnsemble vs each model (paired bootstrap of ΔDice + Wilcoxon):")
ens = scores["Ensemble"]
for m in model_cols:
    if m == "Ensemble":
        continue
    diff = ens - scores[m]
    # paired bootstrap CI of the mean difference
    samples = np.empty(N_BOOTSTRAP)
    for b in range(N_BOOTSTRAP):
        take = rng.integers(0, n, n)
        samples[b] = diff[take].mean()
    lo, hi = np.percentile(samples, [2.5, 97.5])
    try:
        p = wilcoxon(ens, scores[m]).pvalue
    except ValueError:
        p = float("nan")
    better = float(np.mean(ens > scores[m]))
    print(f"  vs {m:22s} ΔDice={diff.mean():+.3f}  95%CI=[{lo:+.3f}, {hi:+.3f}]  "
          f"Wilcoxon p={p:.2e}  ensemble better on {better*100:.0f}% of scenes")


## 6. Area fidelity of the ensemble

Per-scene plume areas of the ensemble against the reference masks: the
median relative area error, the median signed bias and the Pearson
correlation between the predicted and reference areas over the test scenes.

In [ ]:
ens_dir = PRED_DIRS["Ensemble"]
gt_areas, pred_areas = [], []
for scene in df["scene"]:
    gt = load_binary(os.path.join(GT_DIR, scene))
    if gt is None:
        continue
    ppath = os.path.join(ens_dir, scene.replace(".png", "_pred.png"))
    if not os.path.exists(ppath):
        ppath = os.path.join(ens_dir, scene)
    pred = load_binary(ppath, ref_shape=gt.shape)
    if pred is None:
        continue
    gt_areas.append(float(gt.sum()))
    pred_areas.append(float(pred.sum()))

gt_areas = np.asarray(gt_areas)
pred_areas = np.asarray(pred_areas)
rel_err = np.abs(pred_areas - gt_areas) / np.maximum(gt_areas, 1.0)
signed = (pred_areas - gt_areas) / np.maximum(gt_areas, 1.0)
pcc = float(np.corrcoef(gt_areas, pred_areas)[0, 1])

print("Scenes:                       %d" % len(gt_areas))
print("Median relative area error:   %.3f" % np.median(rel_err))
print("Median signed area bias:      %+.3f" % np.median(signed))
print("Area PCC (pred vs reference): %.3f" % pcc)